# RAG: Retrieval-Augmented Generation

> Ask an LLM "what was in yesterday's news" or "what does the company internal document say," and it will either say it doesn't know or fabricate a plausible-sounding answer. Once training is complete, knowledge is frozen — this is a fundamental limitation of LLMs.
>
> In this section, we build RAG (Retrieval-Augmented Generation) from scratch, enabling an LLM to "look up reference material" at generation time and answer questions beyond its training data.

The core problem RAG solves is that an LLM's knowledge cannot be updated. Once training finishes, the model's parameters are fixed — it cannot acquire new information or access private data.

RAG's approach is to first retrieve relevant documents before answering, then concatenate the retrieved content into the prompt, and generate a response based on those documents. The entire process does not modify model parameters — it only changes the model's input. This is sometimes called "training-free knowledge injection."

The complete RAG pipeline has five steps: document chunking, embedding, indexing, retrieval, and generation.

## 1. The Basic Idea of RAG

The core idea behind RAG is simple: **let the model take an "open-book exam."**

```
Traditional LLM:
  User asks a question -> Model answers from memory -> May answer incorrectly

RAG:
  User asks a question -> Retrieve relevant documents -> Stuff them into the prompt -> Model answers with documents in front of it
```

The complete RAG pipeline has five steps:

1. **Chunking**: Split long documents into small segments
2. **Embedding**: Convert each text segment into a vector
3. **Indexing**: Store vectors for fast lookup
4. **Retrieval**: When a user asks a question, find the most relevant document segments
5. **Generation**: Concatenate the retrieved content into the prompt and let the model answer

## 2. Document Chunking

Why do we need to split documents? Because:
- Models have limited context windows: common models range from a few thousand to hundreds of thousands of tokens. Some commercial models have reached the 1M level, but it's still not practical to blindly stuff all materials in. See: [OpenAI GPT-4.1 1M context](https://platform.openai.com/docs/models/gpt-4.1), [GPT-4.1 release notes](https://openai.com/index/gpt-4-1/)
- Retrieval accuracy requires text chunks of "appropriate granularity" — too long and irrelevant content gets mixed in, too short and context is lost

The simplest chunking method is to split by a fixed number of characters:

In [ ]:
# Simulate a document
doc = """Transformer is a neural network architecture based on the self-attention mechanism, proposed by Vaswani et al. in 2017.
Its core innovation is the complete abandonment of recurrent structures (RNN), relying solely on attention mechanisms to model dependencies within sequences.
Transformer consists of an Encoder and a Decoder. The original paper used it for machine translation — the Encoder processes the source language, and the Decoder generates the target language.
The GPT series uses only the Decoder part, generating tokens one by one in an autoregressive manner.
BERT uses only the Encoder part, pre-trained via Masked Language Modeling (MLM).
Modern large language models (such as GPT-4, LLaMA, Qwen) are all based on the Transformer architecture but incorporate many improvements over the original design.
Major improvements include: replacing LayerNorm with RMSNorm, replacing ReLU activation with SwiGLU, and switching from sinusoidal position encoding to RoPE.
Training a large language model requires enormous computational resources and data. For example, LLaMA-65B was trained on 1.4T tokens across 2048 A100 GPUs for approximately 21 days.
The larger the model parameters, the more training data is needed. Chinchilla scaling laws state that for a given compute budget, model parameters and training data should scale proportionally.
During inference, large language models generate tokens one by one in an autoregressive manner. A naive implementation would reprocess all historical tokens at each step; actual deployments typically use KV Cache to reuse historical K/V and avoid full recomputation, though new tokens still need to attend to the cached history.
This is why inference speed is one of the core challenges in LLM deployment. Techniques like KV Cache, FlashAttention, and quantization are all designed to accelerate inference. For more on KV Cache, see the HuggingFace documentation: https://huggingface.co/docs/transformers/main/en/kv_cache.
"""

def chunk_fixed(text, chunk_size=100, overlap=20):
    """Fixed-length chunking with overlap"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        # Try to break at a period
        last_period = chunk.rfind('.')
        if last_period > chunk_size // 2:
            chunk = text[start:start + last_period + 1]
            end = start + last_period + 1
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if len(c) > 10]

chunks = chunk_fixed(doc, chunk_size=120, overlap=20)
print(f"Document length: {len(doc)} characters")
print(f"After chunking: {len(chunks)} segments\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: [{len(chunk)} chars] {chunk[:60]}...")

## 3. Embedding

After chunking, each chunk needs to be converted into a vector. This allows us to mathematically measure "how similar are two pieces of text."

In practice, a dedicated Embedding model (such as BGE or text-embedding-3-small) would be used. Here we use random vectors for simulation — the focus is on understanding the pipeline.

In [ ]:
# Simulate Embedding: use random vectors to represent each chunk
# In practice, a model (like sentence-transformers) would generate semantically meaningful vectors

import numpy as np
import random

def fake_embedding(text, dim=64):
    """Simulate text embedding: generate a pseudo-vector based on character frequency"""
    vec = np.zeros(dim)
    for i, ch in enumerate(text[:dim*3]):
        vec[i % dim] += ord(ch) * 0.01
    # Add a bit of randomness to simulate model output
    vec += np.random.randn(dim) * 0.1
    return vec / np.linalg.norm(vec)  # Normalize

# Generate a vector for each chunk
chunk_vectors = np.array([fake_embedding(c) for c in chunks])

print(f"{len(chunks)} chunks, each embedded into {chunk_vectors.shape[1]} dimensions")
print(f"Vector norm (should be ~1.0 after normalization): {np.linalg.norm(chunk_vectors[0]):.4f}")

## 4. Vector Retrieval

Once we have vectors, retrieval becomes "find the nearest vectors." The most commonly used distance metric is **cosine similarity**:

$$\text{cosine\_sim}(a, b) = \frac{a \cdot b}{|a| \cdot |b|}$$

When vectors are normalized, cosine similarity is simply the dot product.

In [ ]:
# Implement vector retrieval

import numpy as np

def cosine_similarity(a, B):
    """Cosine similarity between query vector a and each row of matrix B"""
    # Assume normalized — dot product suffices
    return B @ a

def retrieve(query, chunks, chunk_vectors, top_k=2):
    """Retrieve the top_k most relevant chunks for a query"""
    query_vec = fake_embedding(query)
    scores = cosine_similarity(query_vec, chunk_vectors)
    top_indices = np.argsort(scores)[-top_k:][::-1]
    results = []
    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": float(scores[idx]),
            "index": int(idx)
        })
    return results

# Test retrieval
query = "How to optimize Transformer inference speed?"
results = retrieve(query, chunks, chunk_vectors, top_k=2)

print(f"Query: {query}\n")
print("Retrieval results:")
for r in results:
    print(f"  [score={r['score']:.4f}] Chunk {r['index']}: {r['chunk'][:70]}...")

## 5. Assembling the Prompt and Generating

After retrieving relevant documents, concatenate them into the prompt. A typical RAG prompt structure:

```
[System Prompt]
Please answer the user's question based on the following reference material. If the reference material does not contain relevant information, say "I don't know."

[Reference Material]
{retrieved document segments}

[User Question]
{user's question}
```

In [ ]:
# Build the RAG prompt

def build_rag_prompt(query, retrieved_chunks):
    """Construct the full RAG prompt"""
    context = "\n\n".join([
        f"[{i+1}] {r['chunk']}" 
        for i, r in enumerate(retrieved_chunks)
    ])
    
    prompt = f"""Please answer the user's question based on the following reference material. If the reference material does not contain relevant information, say "Based on the available material, I cannot answer this."

Reference material:
{context}

User question: {query}"""
    return prompt

prompt = build_rag_prompt(query, results)
print(f"RAG prompt length: {len(prompt)} characters\n")
print(prompt)

## 6. Evaluating Retrieval Quality

RAG effectiveness depends heavily on retrieval quality. If the retrieved content is irrelevant, even the strongest model won't answer well.

Two common metrics:
- **Recall@K**: Out of all relevant documents, how many were retrieved
- **MRR (Mean Reciprocal Rank)**: What rank is the first relevant document at

In [ ]:
# Simulate retrieval quality evaluation

# Assume 5 queries, each with labeled "relevant chunks"
test_queries = [
    {"query": "Who proposed the Transformer?", "relevant": [0]},
    {"query": "Which part of Transformer does GPT use?", "relevant": [1, 2]},
    {"query": "How many GPUs were used to train LLaMA?", "relevant": [5]},
    {"query": "What is Chinchilla scaling law?", "relevant": [6]},
    {"query": "What are the methods for inference acceleration?", "relevant": [8, 9]},
]

def evaluate_retrieval(test_queries, chunks, chunk_vectors, top_k=3):
    """Evaluate retrieval recall and MRR"""
    recall_sum = 0
    mrr_sum = 0
    
    for tq in test_queries:
        results = retrieve(tq['query'], chunks, chunk_vectors, top_k=top_k)
        retrieved_ids = set(r['index'] for r in results)
        relevant_ids = set(tq['relevant'])
        
        # Recall
        if len(relevant_ids) > 0:
            recall = len(retrieved_ids & relevant_ids) / len(relevant_ids)
        else:
            recall = 0
        recall_sum += recall
        
        # MRR: what rank is the first relevant result at
        for rank, r in enumerate(results, 1):
            if r['index'] in relevant_ids:
                mrr_sum += 1.0 / rank
                break
    
    n = len(test_queries)
    return recall_sum / n, mrr_sum / n

recall, mrr = evaluate_retrieval(test_queries, chunks, chunk_vectors, top_k=3)
print(f"Recall@3: {recall:.2f}")
print(f"MRR:      {mrr:.2f}")
print(f"\nNote: We're using random vectors (fake_embedding), so retrieval quality is low")
print(f"In practice with a real Embedding model, Recall@3 typically reaches 0.8-0.95")

## 7. Advanced Optimizations

The basic "chunk -> embed -> retrieve -> generate" pipeline solves 80% of problems. The remaining 20% requires some advanced techniques:

| Technique | What problem it solves | How it works |
|:----------|:----------------------|:-------------|
| **Re-ranker** | Initial retrieval may be inaccurate; use a stronger (but slower) model to re-rank | First retrieve top-20 via vector search, then re-rank with a cross-encoder to top-5 |
| **HyDE** | User queries are short and don't match document wording | Have the model write a "fake answer" first, then use that fake answer for retrieval |
| **Multi-route recall** | Vector retrieval may miss cases where keyword matching is critical | Combine vector retrieval + keyword retrieval (BM25) and merge results |
| **Metadata filtering** | Documents have structured attributes (date, category) | Filter by metadata before retrieval to narrow the search space |
| **Context window compression** | Retrieved content is too large to fit in the context window | Use a model to summarize each chunk, keeping only key information |

**Re-ranker intuition**:
```
Vector retrieval (fast but coarse):
  Query: "LLaMA training cost" -> Returns 20 possibly relevant chunks

Re-ranker (slow but precise):
  Concatenate the query with each chunk, pass through a cross-encoder
  -> Re-rank and take the top-5 most relevant
```

This is like using a search engine to quickly find 20 web pages, then manually scanning the titles to select the 5 most relevant ones.

In [ ]:
# Simulate the effect of a Re-ranker

# Stage 1: Vector retrieval top-5 (simulating coarse filtering)
query = "How much compute is needed to train a large model?"
stage1_results = retrieve(query, chunks, chunk_vectors, top_k=5)

print("=== Stage 1: Vector Retrieval Top-5 ===")
for r in stage1_results:
    print(f"  [{r['score']:.4f}] {r['chunk'][:50]}...")

# Stage 2: Simulate Re-ranker (give higher scores to more relevant results)
def fake_rerank(query, results):
    """Simulate Re-ranker: re-rank based on keyword overlap"""
    query_keywords = set(query.replace('?', '').split())
    for r in results:
        chunk_words = set(r['chunk'].lower().split())
        keyword_overlap = len(query_keywords & chunk_words)
        r['rerank_score'] = r['score'] + keyword_overlap * 0.1
    results.sort(key=lambda x: x['rerank_score'], reverse=True)
    return results

stage2_results = fake_rerank(query, stage1_results[:3])

print(f"\n=== Stage 2: After Re-ranking Top-3 ===")
for r in stage2_results:
    print(f"  [rerank={r['rerank_score']:.4f}] {r['chunk'][:50]}...")

print(f"\nKey observation: The Re-ranker may change the ranking, bringing the most relevant results to the top")

## 8. RAG Architecture in Production

RAG systems in production are much more complex than the demo above, but the core pipeline is the same:

```
                    ┌──────────────┐
                    │  User Query  │
                    └──────┬───────┘
                           │
                    ┌──────▼───────┐
                    │ Query Rewrite │  <- Optional: make the query easier to retrieve
                    └──────┬───────┘
                           │
              ┌────────────┼────────────┐
              │            │            │
        ┌─────▼────┐ ┌────▼────┐ ┌────▼────┐
        │  Vector   │ │  BM25   │ │Knowledge│  <- Multi-route recall
        │ Retrieval │ │ Search  │ │  Graph  │
        └─────┬────┘ └────┬────┘ └────┬────┘
              │            │            │
              └────────────┼────────────┘
                           │
                    ┌──────▼───────┐
                    │  Re-ranker   │  <- Fine ranking
                    └──────┬───────┘
                           │
                    ┌──────▼───────┐
                    │  LLM Gen     │  <- Prompt with retrieved results
                    └──────────────┘
```

Common tools:

| Component | Common Tools |
|:----------|:-------------|
| Vector Database | Chroma, FAISS, Milvus, Pinecone |
| Embedding Model | BGE, text-embedding-3-small, Cohere |
| Re-ranker | bge-reranker, Cohere Rerank |
| Framework | LangChain, LlamaIndex |

## Summary

- RAG's core idea is to let the LLM take an "open-book exam": first retrieve relevant documents, then generate answers based on those documents
- The full pipeline: document chunking -> embedding -> indexing -> retrieval -> prompt assembly -> generation
- Chunking requires balancing granularity: too long mixes in irrelevant content, too short loses context
- Retrieval quality is the key to RAG effectiveness: evaluate with Recall@K and MRR
- Advanced optimizations include Re-ranker, HyDE, multi-route recall, and metadata filtering
- RAG does not change model parameters — it only changes the model's input. This is a form of training-free knowledge injection

## Exercises

**Exercise 1**: Implement a paragraph-based chunking function (split by blank lines or newlines). Compare its results with fixed-length chunking.

Hint: First split by `\n\n`, then perform secondary chunking on paragraphs that are too long.

**Exercise 2**: Modify the `retrieve` function to support dual filtering by top_k and a similarity threshold (only return results with score > threshold).

Hint: After `argsort`, add a threshold check to filter out low-scoring results.

**Exercise 3**: Implement BM25 retrieval (based on term frequency) and compare its results with vector retrieval.

Hint: The core of BM25 is computing IDF (inverse document frequency) for each term, then calculating a relevance score based on term frequency and document length.